In [1]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt

from quantum_systems import BasisSet, ODQD, GeneralOrbitalSystem
from configuration_interaction import CISD

In [2]:
num_fd = 1001
num_basis = 10
x_max = 10
x = np.linspace(-x_max, x_max, num_fd)
dx = abs(x[1] - x[0])

omega = 1
alpha = 1
a = 0.01

In [3]:
class HOPotential:
    def __init__(self, omega=1):
        self.omega = omega

    def __call__(self, x):
        return 0.5 * self.omega**2 * x**2

In [4]:
potential = HOPotential(omega=1)

We use the second order central difference scheme for the Laplacian.
This gives us

\begin{align*}
    \frac{\mathrm{d}^2f}{\mathrm{d}x^2}(x)
    = \frac{f(x + \delta x) - 2 f(x) + f(x - \delta x)}{(\delta x)^2}
\end{align*}

In [5]:
diag_term = 1 / (dx**2) * np.ones(num_fd - 2)
off_diag = -1 / (2 * dx**2) * np.ones(num_fd - 3)

t = np.diag(diag_term) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)

v = potential(x[1:-1])

h = t + np.diag(v)

In [7]:
def shielded_coulomb_operator(x_1, x_2, kappa, a):
    return kappa / np.sqrt((x_1 - x_2) ** 2 + a**2)

In [8]:
u_pq = shielded_coulomb_operator(x[1:-1][None, :], x[1:-1][:, None], alpha, a)

In [9]:
eps, C = scipy.linalg.eigh(h)

# Compare with tridiagonal eigenvalue solver
eps_b, C_b = scipy.linalg.eigh_tridiagonal(diag_term + v, off_diag)
np.testing.assert_allclose(eps, eps_b)
np.testing.assert_allclose(np.abs(C_b), np.abs(C))

In [10]:
C_new = C[:, :num_basis]

In [11]:
h_new = np.einsum("pa, pq, qb -> ab", C_new, h, C_new)

In [12]:
np.diag(h_new)

array([0.4999875 , 1.4999375 , 2.49983749, 3.49968747, 4.49948744,
       5.49923739, 6.49893733, 7.49858723, 8.49818711, 9.49773696])

In [13]:
u_new = np.einsum(
    "pa, qb, pc, qd, pq -> abcd", C_new, C_new, C_new, C_new, u_pq, optimize=True
)

In [15]:
bs = BasisSet(num_basis, dim=1)

bs.h = h_new
bs.u = u_new
bs.s = np.eye(num_basis)

In [16]:
system = GeneralOrbitalSystem(2, bs.copy_basis())

In [17]:
cisd = CISD(system, verbose=True).compute_ground_state()
print(cisd.energies[:10])

Number of states to create: 190
Size of a state in bytes: 8
Time spent setting up CISD space: 2.4522128105163574 sec


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Time spent constructing one-body Hamiltonian: 12.774327993392944 sec
Time spent constructing two-body Hamiltonian: 14.377215147018433 sec
Time spent diagonalizing Hamiltonian: 0.026458024978637695 sec
CISD ground state energy: 2.680009350121367
[2.68000935 2.73107798 2.73107798 2.73107798 3.68529029 3.73157544
 3.73157544 3.73157544 4.53414369 4.6366722 ]


In [18]:
system_o = GeneralOrbitalSystem(
    2, ODQD(num_basis, x_max, num_fd, alpha=alpha, a=a, potential=potential)
)

In [19]:
cisd_o = CISD(system_o, verbose=True).compute_ground_state()
print(cisd_o.energies[:10])

Number of states to create: 190
Size of a state in bytes: 8
Time spent setting up CISD space: 9.703636169433594e-05 sec
Time spent constructing one-body Hamiltonian: 0.01705193519592285 sec
Time spent constructing two-body Hamiltonian: 0.06240415573120117 sec
Time spent diagonalizing Hamiltonian: 0.06779694557189941 sec
CISD ground state energy: 2.6800093501219373
[2.68000935 2.73107798 2.73107798 2.73107798 3.68529029 3.73157544
 3.73157544 3.73157544 4.53414369 4.6366722 ]


In short, the finite-difference basis should be set in the grid basis first. This saves a lot in terms of finding the Coulomb interaction matrix elements as they are sparse in the DVR-sense.